In [19]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

### Load Dataset

In [7]:
vocab_size = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### Padding

In [8]:
max_len = 200

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

### Model

In [9]:
model = Sequential()

model.add(Embedding(vocab_size, 128, input_length=max_len))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### Compile

In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Train

In [11]:
model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 133s 333ms/step - accuracy: 0.8030 - loss: 0.4259 - val_accuracy: 0.8450 - val_loss: 0.3550
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 126s 292ms/step - accuracy: 0.9006 - loss: 0.2499 - val_accuracy: 0.8727 - val_loss: 0.3059
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 114s 292ms/step - accuracy: 0.9347 - loss: 0.1725 - val_accuracy: 0.8653 - val_loss: 0.3301
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 114s 292ms/step - accuracy: 0.9482 - loss: 0.1424 - val_accuracy: 0.8672 - val_loss: 0.3584
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 114s 292ms/step - accuracy: 0.9656 - loss: 0.0964 - val_accuracy: 0.8600 - val_loss: 0.4409


### Evaluate

In [12]:
model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.8600 - loss: 0.4409


[0.4409381151199341, 0.8599600195884705]

In [15]:
import re

def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r'<br />', ' ', text) # Remove <br /> tags
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

In [21]:
# Load the word index to create a tokenizer with the same vocabulary
word_index = imdb.get_word_index()

# Prepare the tokenizer
import tensorflow as tf
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size)
tokenizer.word_index = {k:(v+3) for k,v in word_index.items()} # Adjust indices by 3 for padding, start, unknown tokens
tokenizer.word_index["<PAD>"] = 0
tokenizer.word_index["<START>"] = 1
tokenizer.word_index["<UNK>"] = 2
tokenizer.word_index["<UNUSED>"] = 3 # Optional, if needed

### Prediction

In [22]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    pred = model.predict(padded)

    return "Positive" if pred[0][0] > 0.5 else "Negative"

### Manual Testing

In [23]:
reviews = [
    "The movie was amazing and I loved it",
    "Worst movie ever waste of time"
]

for r in reviews:
    print(r)
    print("Sentiment:", predict_sentiment(r))
    print()

The movie was amazing and I loved it
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Sentiment: Positive

Worst movie ever waste of time
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Sentiment: Negative



### User Input

In [24]:
user_review = input("Enter a review:\n").strip()

print("Sentiment:", predict_sentiment(user_review))

Enter a review:
the movie was amazing and i loved it
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Sentiment: Positive


📌 Code Overview: Sentiment Analysis using RNN/LSTM (IMDb Dataset)

This implementation focuses on building a sentiment analysis model using a Recurrent Neural Network (RNN) variant — Long Short-Term Memory (LSTM) on the inbuilt IMDb dataset. The model classifies movie reviews as positive or negative based on their content.

🔹 1. Dataset Loading

The IMDb dataset is loaded using TensorFlow/Keras. It contains 50,000 movie reviews, split into training and testing datasets. The reviews are already preprocessed and integer-encoded, representing the most frequent words.

🔹 2. Data Preprocessing
The dataset is restricted to a fixed vocabulary size (e.g., top 10,000 words).
Sequences of varying lengths are standardized using padding to ensure uniform input size.
This allows efficient batch processing in the neural network.
🔹 3. Model Architecture

A Sequential deep learning model is built using:

Embedding Layer → Converts integer word indices into dense vector representations
LSTM Layer → Captures sequential dependencies and contextual meaning in text
Dense Layer (Sigmoid Activation) → Outputs probability for binary sentiment classification
🔹 4. Model Compilation

The model is compiled using:

Optimizer: Adam
Loss Function: Binary Crossentropy
Evaluation Metric: Accuracy
🔹 5. Model Training

The model is trained on the training dataset using:

Multiple epochs
Batch processing
Validation using test data
🔹 6. Model Evaluation

The trained model is evaluated on the test dataset to measure accuracy and loss, ensuring its ability to generalize to unseen reviews.

🔹 7. Prediction

The trained model predicts the sentiment of reviews as:

Positive (1)
Negative (0)

Since the dataset is pre-encoded, predictions are made on processed sequences rather than raw text.

🎯 Key Concept

LSTM networks are capable of learning long-term dependencies in sequential data, making them highly effective for natural language processing tasks such as sentiment analysis.

🧠 One-Line Summary

👉 Text (encoded) → Padding → Embedding → LSTM → Sentiment Prediction